In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, lit, to_timestamp,
    hour, dayofweek, datediff, current_date,
    count, avg, stddev, lag, abs as spark_abs,
    current_timestamp, round as spark_round
)
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()
spark.sql("USE fraud_db")

# Read from Bronze — our single source of truth
bronze_df = spark.table("fraud_db.bronze_transactions")

print(f"✅ Bronze records loaded : {bronze_df.count():,}")
print(f"\n📋 Bronze Schema:")
bronze_df.printSchema()

In [0]:

print("🔍 DATA QUALITY REPORT — Bronze Layer")
print("=" * 55)

total = bronze_df.count()

# 1. Null checks on critical fields
print("\n  📌 NULL CHECK:")
for field in ["transaction_id", "customer_id", "amount", 
              "country_code", "is_fraud"]:
    null_count = bronze_df.filter(col(field).isNull()).count()
    status = "✅" if null_count == 0 else "⚠️ "
    print(f"     {status} {field:25s} nulls: {null_count}")

# 2. Duplicate transaction IDs
dup_count = total - bronze_df.select("transaction_id").distinct().count()
print(f"\n  📌 DUPLICATE TRANSACTION IDs : {dup_count}")

# 3. Invalid amounts (negative or zero)
bad_amount = bronze_df.filter(col("amount") <= 0).count()
print(f"  📌 INVALID AMOUNTS (<=0)     : {bad_amount}")

# 4. Invalid hours
bad_hour = bronze_df.filter(
    (col("hour_of_day") < 0) | (col("hour_of_day") > 23)
).count()
print(f"  📌 INVALID HOURS             : {bad_hour}")

# 5. Fraud label check (should only be 0 or 1)
bad_label = bronze_df.filter(
    ~col("is_fraud").isin([0, 1])
).count()
print(f"  📌 INVALID FRAUD LABELS      : {bad_label}")

print("\n" + "=" * 55)
print("  ✅ Quality check complete — We're proceeding to Silver!")

In [0]:
silver_clean = bronze_df \
    .filter(col("amount") > 0) \
    .filter(col("hour_of_day").between(0, 23)) \
    .filter(col("is_fraud").isin([0, 1])) \
    .dropDuplicates(["transaction_id"]) \
    .withColumn(
        "transaction_ts",
        to_timestamp(col("transaction_dt"), "yyyy-MM-dd HH:mm:ss")
    ) \
    .withColumn(
        "amount",
        spark_round(col("amount"), 2)       # ensure 2 decimal places
    ) \
    .withColumn(
        "country_code",
        col("country_code").cast("string")  # explicit cast
    ) \
    .drop("transaction_dt")                 # replaced by transaction_ts

print(f"✅ Records before cleaning : {bronze_df.count():,}")
print(f"✅ Records after cleaning  : {silver_clean.count():,}")
print(f"   Dropped                 : {bronze_df.count() - silver_clean.count()}")
print(f"\n📋 Silver Clean Schema:")
silver_clean.printSchema()

In [0]:
silver_featured = silver_clean \
    .withColumn(
        "is_night_transaction",          # 🌙 Fraud loves late hours
        when(
            (col("hour_of_day") >= 0) & (col("hour_of_day") <= 5),
            lit(1)
        ).otherwise(lit(0))
    ) \
    .withColumn(
        "is_high_amount",                # 💰 Unusually large transaction
        when(col("amount") > 1000, lit(1)).otherwise(lit(0))
    ) \
    .withColumn(
        "is_suspicious_country",         # 🌍 High-risk country codes
        when(
            col("country_code").isin(["NG", "RO", "PK", "XX"]),
            lit(1)
        ).otherwise(lit(0))
    ) \
    .withColumn(
        "is_multiple_attempts",          # 🔐 Multiple login = account takeover
        when(col("login_attempts") > 2, lit(1)).otherwise(lit(0))
    ) \
    .withColumn(
        "is_online_or_atm",              # 💻 Higher fraud channel
        when(
            col("merchant_category").isin(["online", "atm"]),
            lit(1)
        ).otherwise(lit(0))
    ) \
    .withColumn(
        "amount_risk_band",              # 💵 Categorize transaction size
        when(col("amount") < 50,   lit("low"))
        .when(col("amount") < 500, lit("medium"))
        .when(col("amount") < 2000, lit("high"))
        .otherwise(lit("very_high"))
    )

print("✅ Feature Engineering Done!")
print(f"   New columns added: 6 fraud signal features")
print(f"\n📊 Sample — Fraudulent transactions with features:")
display(
    silver_featured
    .filter(col("is_fraud") == 1)
    .select("transaction_id", "amount", "hour_of_day",
            "country_code", "login_attempts",
            "is_night_transaction", "is_high_amount",
            "is_suspicious_country", "is_multiple_attempts",
            "amount_risk_band")
    .limit(10)
)

In [0]:
silver_scored = silver_featured \
    .withColumn(
        "fraud_risk_score",
        (
            col("is_night_transaction") +
            col("is_high_amount") +
            col("is_suspicious_country") +
            col("is_multiple_attempts") +
            col("is_online_or_atm")
        ).cast("integer")
    ) \
    .withColumn(
        "risk_label",
        when(col("fraud_risk_score") >= 4, lit("CRITICAL"))
        .when(col("fraud_risk_score") == 3, lit("HIGH"))
        .when(col("fraud_risk_score") == 2, lit("MEDIUM"))
        .when(col("fraud_risk_score") == 1, lit("LOW"))
        .otherwise(lit("SAFE"))
    )

print("🎯 FRAUD RISK SCORE DISTRIBUTION:")
silver_scored.groupBy("risk_label", "is_fraud") \
    .count() \
    .orderBy("risk_label", "is_fraud") \
    .show()

print("\n📊 Avg Risk Score by Actual Fraud Label:")
silver_scored.groupBy("is_fraud") \
    .avg("fraud_risk_score") \
    .withColumnRenamed("avg(fraud_risk_score)", "avg_risk_score") \
    .withColumn("avg_risk_score", spark_round(col("avg_risk_score"), 2)) \
    .show()

In [0]:

spark.sql("DROP TABLE IF EXISTS fraud_db.silver_transactions")

silver_scored.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fraud_db.silver_transactions")

count = spark.table("fraud_db.silver_transactions").count()

print(f"✅ Silver Layer Written!")
print(f"   📦 Records    : {count:,}")
print(f"   📋 Table      : fraud_db.silver_transactions")
print(f"\n📊 Final Silver Schema:")
spark.table("fraud_db.silver_transactions").printSchema()